In [1]:
import optax.projections
%load_ext autoreload
%autoreload 2

# XLA_PYTHON_CLIENT_PREALLOCATE = False
import jax
import jax.numpy as jnp
import equinox as eqx

import numpy as np
import matplotlib.pyplot as plt

from qdots_qll.distributions import Distribution, update_particles_locations, update_weights

import numpy as np
import qutip as qt
import matplotlib.pyplot as plt

from sklearn.metrics import mean_squared_error

from jax import jit
from jax.scipy.linalg import expm

from qdots_qll.models.single_dot_weak_coupling_GAME import *

from qdots_qll.resamplers import LWResamplerBounds

boundaries = jnp.array([
    [0.01, 0.9],
    [0.01, 0.9],
    [0.001, 0.22],
    [-0.9, -0.01],
])

resampler = LWResamplerBounds(a=0.95, parameters_bounds=boundaries)


In [2]:
def det_fim(model, *args, **kwargs):
    return jnp.linalg.det(model.fim(*args, **kwargs))


def choose_given_p(key, discrete_distribution):
    outcomes = jnp.arange(discrete_distribution.shape[0])
    return jax.random.choice(key, outcomes, p=discrete_distribution)



In [3]:
no_particles = 250
no_rv = 4

print(true_parameters)
seed = 1
key = jax.random.PRNGKey(seed)
key, subkey = jax.random.split(key)

model = SingleDotWeakCouplingGAME()

no_initial_states = 4
no_measurement_basis = 3
dist_initial_state = jnp.ones(no_initial_states) / no_initial_states
dist_measurement_basis = jnp.ones(no_measurement_basis) / no_measurement_basis

initial_covariance = jnp.diagflat(jnp.ones(4) * 0.05) ** 2
# particles_locations = jax.random.multivariate_normal(subkey,
#                                                      mean=true_parameters,
#                                                      cov=initial_covariance,
#                                                      shape=(no_particles,))

from tensorflow_probability.substrates import jax as tfp

mus = boundaries.mean(axis=1)
sigmas = jnp.abs((boundaries[:, 0] - boundaries[:, 1]) / (2 * 4))

key, subkey = jax.random.split(key)
particles_locations = tfp.distributions.TruncatedNormal(loc=mus, scale=sigmas, low=boundaries[:, 0],
                                                        high=boundaries[:, 1]).sample(
    seed=subkey, sample_shape=no_particles)

weights = jnp.ones(no_particles) / no_particles

distribution = Distribution(particles_locations, weights)


In [4]:
i, j = 0, 3

plt.plot(distribution.particles_locations[:, i], distribution.particles_locations[:, j], '.', label='old')

key, subkey = jax.random.split(key)

key, new_weights, new_particles_locations = resampler.resample(key, distribution.particles_locations,
                                                               distribution.weights).values()

new_distribution = update_particles_locations(distribution, new_particles_locations)
new_distribution = update_weights(new_distribution, new_weights)

plt.plot(new_distribution.particles_locations[:, i], new_distribution.particles_locations[:, j], '.', label='new')

plt.axhline(boundaries[j, 0])
plt.axhline(boundaries[j, 1])
plt.axvline(boundaries[i, 0])
plt.axvline(boundaries[i, 1])
plt.legend()
plt.show()

In [5]:
from qdots_qll.exp_design import MaxDetFimExpDesign

expdesign = MaxDetFimExpDesign(t_min=0.01, t_max=45., sgd_iter=10, lr=0.05)

times = jnp.linspace(0, 50, 1000)

det_fim_times = jax.vmap(
    lambda t: jnp.linalg.det(model.fim(particle=distribution.ev(), t=t, prob_initial_state=dist_initial_state,
                                       prob_measurement_basis=dist_measurement_basis)))(
    times)

new_t = expdesign.generate_time(key=subkey, particles_locations=distribution.particles_locations,
                                weights=distribution.weights,
                                model=model, prob_initial_state=dist_initial_state,
                                prob_measurement_basis=dist_measurement_basis)

print(new_t)
key, subkey = jax.random.split(key)

plt.plot(times, det_fim_times)
plt.axvline(new_t)

In [6]:
def loss_function_probs(params, model, **kwargs):
    p_initial_state = params["state"]
    p_measurement_basis = params["measurement"]

    loss = -1 * jnp.linalg.det(
        model.fim(prob_initial_state=p_initial_state, prob_measurement_basis=p_measurement_basis, **kwargs))
    return loss


In [7]:
import optax

optimizer = optax.sgd(learning_rate=0.1)
params = {"state": dist_initial_state, "measurement": dist_measurement_basis}
opt_state = optimizer.init(params)

In [8]:


# for _ in range(15):
#     grads = jax.grad(loss_function_probs)(params, model, t=times[200], particle=true_parameters)
#     # print(grads)
# 
#     updates, opt_state = optimizer.update(grads, opt_state)
#     params = optax.apply_updates(params, updates)
#     params['state'] = optax.projections.projection_simplex(params['state'])
#     params['measurement'] = optax.projections.projection_simplex(params['measurement'])
#     print(loss_function_probs(params, model, t=times[200], particle=true_parameters))
# 
# params


In [9]:
from qdots_qll.exp_design import OptimizeInitialStateMeasurements

In [10]:
popt = OptimizeInitialStateMeasurements(iter=5, lr=0.05)
expdesign = MaxDetFimExpDesign(t_min=0.01, t_max=45., sgd_iter=10, lr=0.05)

In [92]:
new_p_state, new_p_measurement = popt.optimize_probability_distribution(loss_function=loss_function_probs,
                                                                        dist_initial_state=dist_initial_state,
                                                                        dist_measurement_basis=dist_measurement_basis,
                                                                        model=model, t=times[300],
                                                                        particle=true_parameters)

In [29]:
loss_function_probs((dist_initial_state, dist_measurement_basis), model, t=times[400], particle=true_parameters)

In [ ]:
loss_f_initial_state = lambda p_state, p_basis: -1 * jnp.linalg.det(
    model.fim(prob_initial_state=p_state, prob_measurement_basis=p_basis, **kwargs))

In [93]:

times = jnp.linspace(0, 50, 1000)

det_fim_times_old = jax.vmap(
    lambda t: jnp.linalg.det(model.fim(particle=true_parameters, t=t, prob_initial_state=dist_initial_state,
                                       prob_measurement_basis=dist_measurement_basis)))(
    times)

det_fim_times_new = jax.vmap(
    lambda t: jnp.linalg.det(model.fim(particle=true_parameters, t=t, prob_initial_state=new_p_state,
                                       prob_measurement_basis=new_p_measurement)))(
    times)

plt.plot(times, det_fim_times_new, label='new')

plt.plot(times, det_fim_times_old, label='old')
plt.legend()
plt.show()


In [489]:
true_parameters

In [310]:
dist_initial_state

In [427]:
new_dist_initial_state

In [11]:
from qdots_qll.exp_design import OptimizeInitialStateMeasurements, MaxDetFimExpDesign


boundaries = jnp.array([
    [0.1, 0.5],
    [0.1, 0.5],
    [0.01, 0.2],
    [-0.5, -0.01],
])



no_particles = 100
key = jax.random.PRNGKey(seed=4)

popt = OptimizeInitialStateMeasurements(iter=4, lr=0.005)
expdesign = MaxDetFimExpDesign(t_min=0.01, t_max=45., sgd_iter=4, lr=0.01)
resampler = LWResamplerBounds(a=0.98, parameters_bounds=boundaries)

mus = boundaries.mean(axis=1)
sigmas = jnp.abs((boundaries[:, 0] - boundaries[:, 1]) / (2 * 1))

key, subkey = jax.random.split(key)
particles_locations = tfp.distributions.TruncatedNormal(loc=mus, scale=sigmas, low=boundaries[:, 0],
                                                        high=boundaries[:, 1]).sample(
    seed=subkey, sample_shape=no_particles)

weights = jnp.ones(no_particles) / no_particles

distribution = Distribution(particles_locations, weights)

new_dist_initial_state, new_dist_measurement_basis = dist_initial_state, dist_measurement_basis

times_list = []
cov_list = []
outcomes_list = []


In [138]:

for _ in range(50):
    key, subkey = jax.random.split(key)

    t = eqx.filter_jit(expdesign.generate_time)(key=subkey, particles_locations=distribution.particles_locations,
                                                weights=distribution.weights,
                                                model=model, prob_initial_state=new_dist_initial_state,
                                                prob_measurement_basis=new_dist_measurement_basis)
    times_list.append(t)
    new_dist_initial_state, new_dist_measurement_basis = eqx.filter_jit(popt.optimize_probability_distribution)(
        dist_initial_state=new_dist_initial_state,
        dist_measurement_basis=new_dist_measurement_basis,
        model=model, t=t,
        particle=distribution.ev())

    key, subkey = jax.random.split(key)
    chosen_initial_state = jax.random.choice(subkey, jnp.arange(no_initial_states), p=new_dist_initial_state)

    key, subkey = jax.random.split(key)
    chosen_basis = jax.random.choice(subkey, jnp.arange(no_measurement_basis), p=new_dist_measurement_basis)

    key, subkey = jax.random.split(key)
    outcome = eqx.filter_jit(model.generate_data)(subkey, true_parameters, t, chosen_initial_state, chosen_basis)
    outcomes_list.append(outcome)

    lkl_particles = eqx.filter_jit(jax.vmap((lambda particle: ((
        model.likelihood_particle_with_basis_initial_state)(particle, t, new_dist_initial_state,
                                                            new_dist_measurement_basis)[
        *outcome]))))(
        distribution.particles_locations)

    distribution = eqx.filter_jit(update_weights)(distribution, lkl_particles)

    # print(distribution.check_resampling())
    # print(distribution.ESS())
    est_cov = jnp.diag(distribution.cov())
    cov_list.append(est_cov)
    print(est_cov)
    if distribution.check_resampling():
        print("Resampling")
        aux = (eqx.filter_jit(resampler.resample)(key, distribution.particles_locations,
                                                  distribution.weights))

        key, new_weights, new_particles_locations = aux['key'], aux['weights'], aux['particles_locations']

        distribution = Distribution(new_particles_locations, new_weights)

times = jnp.linspace(0, 50, 1000)

det_fim_times_old = jax.vmap(
    lambda t: jnp.linalg.det(model.fim(particle=true_parameters, t=t, prob_initial_state=new_dist_initial_state,
                                       prob_measurement_basis=new_dist_measurement_basis)))(
    times)

# det_fim_times_old = jax.vmap(
#     lambda t: jnp.linalg.det(model.fim(particle=true_parameters, t=t, prob_initial_state=dist_initial_state,
#                                        prob_measurement_basis=dist_measurement_basis)))(
#     times)

print(distribution.ev())
print(true_parameters)


det_fim_times_new = jax.vmap(
    lambda t: jnp.linalg.det(model.fim(particle=distribution.ev(), t=t, prob_initial_state=new_dist_initial_state,
                                       prob_measurement_basis=new_dist_measurement_basis)))(
    times)

plt.plot(times, det_fim_times_new, label='ev')

plt.plot(times, det_fim_times_old, label='true pars')
plt.legend()
plt.show()

print(new_dist_initial_state.round(3))
print(new_dist_measurement_basis.round(3))


plt.hist(times_list, bins=100)
plt.show()



plt.plot(cov_list, '-.')
plt.loglog()
plt.show()





In [139]:

det_fim_times_old = jax.vmap(
    lambda t: jnp.linalg.det(model.fim(particle=true_parameters, t=t, prob_initial_state=dist_initial_state,
                                       prob_measurement_basis=dist_measurement_basis)))(
    times)

# det_fim_times_old = jax.vmap(
#     lambda t: jnp.linalg.det(model.fim(particle=true_parameters, t=t, prob_initial_state=dist_initial_state,
#                                        prob_measurement_basis=dist_measurement_basis)))(
#     times)

det_fim_times_new = jax.vmap(
    lambda t: jnp.linalg.det(model.fim(particle=distribution.ev(), t=t, prob_initial_state=dist_initial_state,
                                       prob_measurement_basis=dist_measurement_basis)))(
    times)

plt.plot(times, det_fim_times_new, label='ev')

plt.plot(times, det_fim_times_old, label='true pars')
plt.legend()
plt.show()

In [140]:
i, j = 0, 2

plt.plot(distribution.particles_locations[:, i], distribution.particles_locations[:, j], '.', label='particles')

plt.plot(true_parameters[i], true_parameters[j], '*', label='true')

plt.axhline(boundaries[j, 0])
plt.axhline(boundaries[j, 1])
plt.axvline(boundaries[i, 0])
plt.axvline(boundaries[i, 1])
plt.legend()
plt.show()

In [141]:
new_dist_initial_state

In [143]:
len(outcomes_list)

In [17]:
times_list

# Studying regions of lkl

In [18]:
boundaries

In [19]:
boundaries[:, 0]

In [25]:
def lkl_particle_outcome_time(particle, t, outcome):
    lkl = model.likelihood_particle(particle, t)
    return lkl[*outcome]

def log_lkl_particle_over_data(particle, array_t, array_outcomes):
    return jnp.log(jax.vmap(lkl_particle_outcome_time, in_axes=(None, 0, 0))(particle, array_t, array_outcomes)).sum()

In [170]:
uniformdist = tfp.distributions.Uniform(low=boundaries[:, 0], high=boundaries[:, 1])

uniform_points = uniformdist.sample(seed=subkey, sample_shape=(50000, ))

log_lkl_uniform_points_over_data = jax.vmap(log_lkl_particle_over_data, in_axes=(0, None, None))(uniform_points, np.array(times_list), np.array(outcomes_list))

In [80]:
# fig, ax = plt.subplots(4, 4, figsize=(12, 12))
# 
# for i in range(4):
#     for j in range(4):
#         if i == j:
#             continue
#         
#         ax[i,j].plot(uniform_points[:, i], uniform_points[:, j], '.',  )

In [82]:
# import pandas as pd
# import seaborn as sns
# 
# 
# colnames = ['a', 'b', 'c', 'd', 'loglkl']
# 
# d_aux = np.hstack((np.array(uniform_points), np.array(log_lkl_uniform_points_over_data)[:, None]))
# 
# df = pd.DataFrame(d_aux, columns=colnames)





In [145]:
from scipy.interpolate import griddata



In [171]:
data = np.hstack((np.array(uniform_points), np.array(log_lkl_uniform_points_over_data)[:, None]))


In [147]:
data.shape

In [188]:
# discrete_steps = 100
i, j = 0, 1
x = data[: ,i]
y = data[: , j]
z = data[:, -1]
grid_x, grid_y = np.mgrid[min(x):max(x):1000j, min(y):max(y):1000j]
grid_z = griddata((x, y), z, (grid_x, grid_y), method='nearest')





In [189]:
plt.figure(figsize=(8, 6))
contour = plt.contour(grid_x, grid_y, grid_z, cmap='viridis')
plt.clabel(contour, inline=True, fontsize=8)
plt.title('Contour Lines')
plt.xlabel('X Coordinate')
plt.ylabel('Y Coordinate')
plt.colorbar(contour, label='Value of z')
plt.show()

In [182]:
plt.figure(figsize=(8, 6))
contour = plt.contourf(grid_x, grid_y, grid_z, cmap='viridis')
plt.colorbar(contour, label='Value of z')
plt.title('Contour Map')
plt.xlabel('X Coordinate')
plt.ylabel('Y Coordinate')

plt.plot(true_parameters[i], true_parameters[j], '*', label='true', ms=20)
plt.show()

In [177]:
plt.figure(figsize=(8, 6))
plt.imshow(grid_z.T, extent=(min(x), max(x), min(y), max(y)), origin='lower', cmap='rainbow')
plt.colorbar(label='Value of z')
plt.title('Heatmap')
plt.xlabel('X Coordinate')
plt.ylabel('Y Coordinate')
plt.show()

In [178]:
plt.figure(figsize=(8, 6))
contour = plt.contourf(grid_x, grid_y, grid_z, cmap='viridis')
plt.colorbar(contour, label='Value of z')
plt.title('Contour Map')
plt.xlabel('X Coordinate')
plt.ylabel('Y Coordinate')
plt.show()